In [1]:
import pandas as pd
import numpy as np
from lstm_encoder_decoder import lstm_seq2seq
import torch

: 

In [ ]:
file_path = "mid_datav1.csv"

In [ ]:
df = pd.read_csv(file_path)

In [ ]:
df.head()

,Unnamed: 0,MSG_TYPE,MMSI,NAME,IMO_NUMBER,CALL_SIGN,LAT_AVG,LON_AVG,PERIOD,SPEED_KNOTS,...,time_diff,cog_diff,new_voyage,voyage_id,geometry,in_channel_north,in_channel_south,location,num_pings,include
0,67,1,205089000,DICKENS,9898553.0,ONKZ,21.908479,-76.971564,2023-03-01 08:25:00,13.4,...,1200.0,2.0,True,13_205089000,POINT (-76.971564 21.908479),False,False,east,5,False
1,68,1,205089000,DICKENS,9898553.0,ONKZ,21.901958,-76.959594,2023-03-01 08:30:00,13.2,...,300.0,4.3,False,13_205089000,POINT (-76.959594 21.901958),False,False,east,5,False
2,69,1,205089000,DICKENS,9898553.0,ONKZ,21.895055,-76.938755,2023-03-01 08:35:00,13.2,...,300.0,6.5,False,13_205089000,POINT (-76.938755 21.895055),False,False,east,5,False
3,70,1,205089000,DICKENS,9898553.0,ONKZ,21.876328,-76.888575,2023-03-01 08:45:00,13.4,...,600.0,2.0,False,13_205089000,POINT (-76.888575 21.876328),False,False,east,5,False
4,71,1,205089000,DICKENS,9898553.0,ONKZ,21.872654,-76.878911,2023-03-01 08:50:00,13.4,...,300.0,0.2,False,13_205089000,POINT (-76.878911 21.872654),False,False,east,5,False


In [ ]:
print(df.columns)

Index(['Unnamed: 0', 'MSG_TYPE', 'MMSI', 'NAME', 'IMO_NUMBER', 'CALL_SIGN',
       'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS', 'COG_DEG', 'HEADING_DEG',
       'NAV_STATUS', 'NAV_SENSOR', 'SHIP_AND_CARGO_TYPE', 'DRAUGHT',
       'DRAUGHT.1', 'DIM_BOW', 'DIM_STERN', 'DIM_PORT', 'DIM_STARBOARD',
       'DESTINATION', 'MMSI_COUNTRY_CD', 'RECEIVER', 'BEAM', 'LENGTH',
       'CHANNEL_SIDE', 'time_diff', 'cog_diff', 'new_voyage', 'voyage_id',
       'geometry', 'in_channel_north', 'in_channel_south', 'location',
       'num_pings', 'include'],
      dtype='str')


In [ ]:
df = df[['MMSI', 'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS', 'COG_DEG', 'HEADING_DEG', 'voyage_id']]

In [ ]:
df['TIME'] = pd.to_datetime(df['PERIOD'])
df = df.rename(columns = {'LAT_AVG':'LAT', 'LON_AVG':'LON', 'SPEED_KNOTS':'SPEED', 'COG_DEG':'COG', 'HEADING_DEG':'HEADING'})

print(df.dtypes)

MMSI                  int64
LAT                 float64
LON                 float64
PERIOD                  str
SPEED               float64
COG                 float64
HEADING             float64
voyage_id               str
TIME         datetime64[us]
dtype: object


In [ ]:
print(len(df))
df = df.dropna()

44521


In [ ]:
print(len(df))

42883


In [ ]:
df.head()

,MMSI,LAT,LON,PERIOD,SPEED,COG,HEADING,voyage_id,TIME
0,205089000,21.908479,-76.971564,2023-03-01 08:25:00,13.4,121.0,119.0,13_205089000,2023-03-01 08:25:00
1,205089000,21.901958,-76.959594,2023-03-01 08:30:00,13.2,116.7,111.0,13_205089000,2023-03-01 08:30:00
2,205089000,21.895055,-76.938755,2023-03-01 08:35:00,13.2,110.2,108.0,13_205089000,2023-03-01 08:35:00
3,205089000,21.876328,-76.888575,2023-03-01 08:45:00,13.4,112.2,109.0,13_205089000,2023-03-01 08:45:00
4,205089000,21.872654,-76.878911,2023-03-01 08:50:00,13.4,112.0,108.0,13_205089000,2023-03-01 08:50:00


In [ ]:
df['dt'] = df['TIME'].diff().dt.total_seconds()
df = df.dropna().reset_index(drop=True)


In [ ]:
df.head()

,MMSI,LAT,LON,PERIOD,SPEED,COG,HEADING,voyage_id,TIME,dt
0,205089000,21.901958,-76.959594,2023-03-01 08:30:00,13.2,116.7,111.0,13_205089000,2023-03-01 08:30:00,300.0
1,205089000,21.895055,-76.938755,2023-03-01 08:35:00,13.2,110.2,108.0,13_205089000,2023-03-01 08:35:00,300.0
2,205089000,21.876328,-76.888575,2023-03-01 08:45:00,13.4,112.2,109.0,13_205089000,2023-03-01 08:45:00,600.0
3,205089000,21.872654,-76.878911,2023-03-01 08:50:00,13.4,112.0,108.0,13_205089000,2023-03-01 08:50:00,300.0
4,205089000,22.547895,-78.097770,2023-09-27 14:10:00,12.9,121.1,121.0,20_205089000,2023-09-27 14:10:00,18163200.0


In [ ]:
df['cog_rad'] = np.deg2rad(df['COG'])
df['cog_sin'] = np.sin(df['cog_rad'])
df['cog_cos'] = np.cos(df['cog_rad'])

df['hdg_rad'] = np.deg2rad(df['HEADING'])
df['hdg_sin'] = np.sin(df['hdg_rad'])
df['hdg_cos'] = np.cos(df['hdg_rad'])


In [ ]:
df['dlat'] = df['LAT'].diff()
df['dlon'] = df['LON'].diff()

df = df.dropna().reset_index(drop=True)


In [ ]:
df.head()

,MMSI,LAT,LON,PERIOD,SPEED,COG,HEADING,voyage_id,TIME,dt,cog_rad,cog_sin,cog_cos,hdg_rad,hdg_sin,hdg_cos,dlat,dlon
0,205089000,21.895055,-76.938755,2023-03-01 08:35:00,13.2,110.2,108.0,13_205089000,2023-03-01 08:35:00,300.0,1.923353,0.938493,-0.345298,1.884956,0.951057,-0.309017,-0.006903,0.020839
1,205089000,21.876328,-76.888575,2023-03-01 08:45:00,13.4,112.2,109.0,13_205089000,2023-03-01 08:45:00,600.0,1.958259,0.925871,-0.377841,1.902409,0.945519,-0.325568,-0.018727,0.050180
2,205089000,21.872654,-76.878911,2023-03-01 08:50:00,13.4,112.0,108.0,13_205089000,2023-03-01 08:50:00,300.0,1.954769,0.927184,-0.374607,1.884956,0.951057,-0.309017,-0.003674,0.009664
3,205089000,22.547895,-78.097770,2023-09-27 14:10:00,12.9,121.1,121.0,20_205089000,2023-09-27 14:10:00,18163200.0,2.113594,0.856267,-0.516533,2.111848,0.857167,-0.515038,0.675241,-1.218859
4,205089000,22.523830,-78.055422,2023-09-27 14:20:00,12.9,121.6,122.0,20_205089000,2023-09-27 14:20:00,600.0,2.122320,0.851727,-0.523986,2.129302,0.848048,-0.529919,-0.024065,0.042348


In [ ]:
vessels = df['MMSI'].unique()

splitter = int(len(vessels) * 0.9)

train_voyages = vessels[:splitter]
test_voyages  = vessels[splitter:]

df_train = df[df['MMSI'].isin(train_voyages)].copy()
df_test  = df[df['MMSI'].isin(test_voyages)].copy()


print(len(df_train)/len(df))
print(len(df_test)/len(df))
print(len(df_train))
print(len(df_test))



0.8372006249854248
0.16279937501457523
35900
6981


In [ ]:
vessels = df['MMSI'].unique()

# Shuffle for randomness (VERY important)
rng = np.random.default_rng(42)
rng.shuffle(vessels)

n_total = len(vessels)

# 10% test
test_split = int(n_total * 0.9)

train_val_vessels = vessels[:test_split]
test_vessels      = vessels[test_split:]

# Now split train_val into train + val
val_frac_of_train = 0.2   # 20% of the 90%
val_split = int(len(train_val_vessels) * (1 - val_frac_of_train))

train_vessels = train_val_vessels[:val_split]
val_vessels   = train_val_vessels[val_split:]

# Create DataFrames
df_train = df[df['MMSI'].isin(train_vessels)].copy()
df_val   = df[df['MMSI'].isin(val_vessels)].copy()
df_test  = df[df['MMSI'].isin(test_vessels)].copy()

# Diagnostics
print("Train %:", len(df_train)/len(df))
print("Val %:",   len(df_val)/len(df))
print("Test %:",  len(df_test)/len(df))

print("Train rows:", len(df_train))
print("Val rows:",   len(df_val))
print("Test rows:",  len(df_test))

Train %: 0.7505188778246776
Val %: 0.1706583335276696
Test %: 0.07882278864765281
Train rows: 32183
Val rows: 7318
Test rows: 3380


In [ ]:
def make_windows_from_df(
    df: pd.DataFrame,
    feature_cols,
    target_cols,
    seq_len: int = 20,
    target_len: int = 5,
    mmsi_col: str = "MMSI",
    time_col: str = "TIME"
):
    """
    Returns:
      X: torch.FloatTensor  (seq_len,  N, n_features)
      Y: torch.FloatTensor  (target_len, N, n_targets)
    """
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])

    X_list = []
    Y_list = []

    for mmsi, g in df.groupby(mmsi_col):
        g = g.sort_values(time_col).reset_index(drop=True)

        feats = g[feature_cols].to_numpy(dtype=np.float32)
        targs = g[target_cols].to_numpy(dtype=np.float32)

        n = len(g)
        if n < seq_len + target_len:
            continue

        for i in range(n - (seq_len + target_len) + 1):
            X_list.append(feats[i : i + seq_len])
            Y_list.append(targs[i + seq_len : i + seq_len + target_len])

    if len(X_list) == 0:
        raise ValueError("No windows created. Increase data or reduce seq_len/target_len.")

    X = torch.from_numpy(np.stack(X_list, axis=0))  # (N, seq_len, n_features)
    Y = torch.from_numpy(np.stack(Y_list, axis=0))  # (N, target_len, 2)

    # Your model expects (seq_len, batch, features)
    X = X.permute(1, 0, 2).contiguous()  # (seq_len, N, n_features)
    Y = Y.permute(1, 0, 2).contiguous()  # (target_len, N, 2)

    return X, Y

In [ ]:
feature_cols = ["LAT","LON","SPEED","dt","cog_sin","cog_cos","hdg_sin","hdg_cos"]
target_cols  = ["LAT","LON"]

seq_len = 20
target_len = 5

X_train, Y_train = make_windows_from_df(df_train, feature_cols, target_cols, seq_len=seq_len, target_len=target_len)
X_test,  Y_test  = make_windows_from_df(df_test,  feature_cols, target_cols, seq_len=seq_len, target_len=target_len)

print("X_train:", X_train.shape, "Y_train:", Y_train.shape)
print("X_test: ", X_test.shape,  "Y_test: ", Y_test.shape)

X_train: torch.Size([20, 18478, 8]) Y_train: torch.Size([5, 18478, 2])
X_test:  torch.Size([20, 1624, 8]) Y_test:  torch.Size([5, 1624, 2])


In [ ]:
import math

def rollout_recursive(model, X, target_len, batch_size=64):
    """
    X: (seq_len, N, n_features)
    returns preds: (target_len, N, 2)
    """
    device = next(model.parameters()).device
    X = X.to(device)

    seq_len, N, _ = X.shape
    preds = torch.zeros(target_len, N, 2, device=device)

    lat_i, lon_i = 0, 1
    n_batches = math.ceil(N / batch_size)

    model.eval()
    with torch.no_grad():
        for b in range(n_batches):
            start = b * batch_size
            end = min(start + batch_size, N)
            bs = end - start

            xb = X[:, start:end, :]  # (seq_len, bs, n_features)

            h0, c0 = model.encoder.init_hidden(bs)
            enc_hidden = (h0.to(device), c0.to(device))

            _, enc_hidden = model.encoder(xb)

            dec_in = xb[-1, :, [lat_i, lon_i]]  # (bs, 2)
            dec_hid = enc_hidden

            outb = torch.zeros(target_len, bs, 2, device=device)
            for t in range(target_len):
                dec_out, dec_hid = model.decoder(dec_in, dec_hid)
                outb[t] = dec_out
                dec_in = dec_out

            preds[:, start:end, :] = outb

    return preds

In [ ]:
EARTH_R_M = 6371000.0

def haversine_meters_torch(lat1, lon1, lat2, lon2):
    lat1 = torch.deg2rad(lat1); lon1 = torch.deg2rad(lon1)
    lat2 = torch.deg2rad(lat2); lon2 = torch.deg2rad(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = torch.sin(dlat/2)**2 + torch.cos(lat1)*torch.cos(lat2)*torch.sin(dlon/2)**2
    c = 2 * torch.atan2(torch.sqrt(a), torch.sqrt(1 - a))
    return EARTH_R_M * c

def metrics_meters(pred_latlon, true_latlon):
    pred_lat, pred_lon = pred_latlon[..., 0], pred_latlon[..., 1]
    true_lat, true_lon = true_latlon[..., 0], true_latlon[..., 1]
    d = haversine_meters_torch(pred_lat, pred_lon, true_lat, true_lon)  # (target_len, N)

    mae = d.mean().item()
    mse = (d**2).mean().item()
    rmse = float(np.sqrt(mse))

    per_step_mae = d.mean(dim=1).detach().cpu().numpy()
    per_step_rmse = torch.sqrt((d**2).mean(dim=1)).detach().cpu().numpy()

    return mae, mse, rmse, per_step_mae, per_step_rmse

def rollout_recursive(model, X, target_len, batch_size=64):
    device = next(model.parameters()).device
    X = X.to(device)

    seq_len, N, _ = X.shape
    preds = torch.zeros(target_len, N, 2, device=device)

    lat_i, lon_i = 0, 1
    n_batches = math.ceil(N / batch_size)

    model.eval()
    with torch.no_grad():
        for b in range(n_batches):
            start = b * batch_size
            end = min(start + batch_size, N)
            bs = end - start

            xb = X[:, start:end, :]

            h0, c0 = model.encoder.init_hidden(bs)
            enc_hidden = (h0.to(device), c0.to(device))

            _, enc_hidden = model.encoder(xb)

            dec_in = xb[-1, :, [lat_i, lon_i]]
            dec_hid = enc_hidden

            outb = torch.zeros(target_len, bs, 2, device=device)
            for t in range(target_len):
                dec_out, dec_hid = model.decoder(dec_in, dec_hid)
                outb[t] = dec_out
                dec_in = dec_out

            preds[:, start:end, :] = outb

    return preds

In [ ]:
target_len_list = [1, 5, 10, 15, 20]
seq_len = 20

X_train_dict, Y_train_dict = {}, {}
X_val_dict,   Y_val_dict   = {}, {}
X_test_dict,  Y_test_dict  = {}, {}

for tl in target_len_list:
    X_train_dict[tl], Y_train_dict[tl] = make_windows_from_df(
        df_train, feature_cols, target_cols,
        seq_len=seq_len, target_len=tl
    )

    X_val_dict[tl], Y_val_dict[tl] = make_windows_from_df(
        df_val, feature_cols, target_cols,
        seq_len=seq_len, target_len=tl
    )

    X_test_dict[tl], Y_test_dict[tl] = make_windows_from_df(
        df_test, feature_cols, target_cols,
        seq_len=seq_len, target_len=tl
    )

# sanity check
for tl in target_len_list:
    print(tl, X_train_dict[tl].shape, Y_train_dict[tl].shape)

1 torch.Size([20, 19827, 8]) torch.Size([1, 19827, 2])
5 torch.Size([20, 18478, 8]) torch.Size([5, 18478, 2])
10 torch.Size([20, 17025, 8]) torch.Size([10, 17025, 2])
15 torch.Size([20, 15783, 8]) torch.Size([15, 15783, 2])
20 torch.Size([20, 14742, 8]) torch.Size([20, 14742, 2])


In [ ]:
import itertools
from pathlib import Path

teacher_forcing_ratio_list = [.2, .3, .4, .5, .6, .7]
learning_rate_list = [.01, .04, .06, .08, .1]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out_dir = Path("grid_models")
out_dir.mkdir(exist_ok=True)

rows = []

for target_len, tfr, lr in itertools.product(target_len_list, teacher_forcing_ratio_list, learning_rate_list):

    X_train = X_train_dict[target_len]
    Y_train = Y_train_dict[target_len]
    X_val   = X_val_dict[target_len]
    Y_val   = Y_val_dict[target_len]
    X_test  = X_test_dict[target_len]
    Y_test  = Y_test_dict[target_len]

    print(f"\nCombo: target_len={target_len}, tfr={tfr}, lr={lr}")

    model = lstm_seq2seq(input_size=X_train.shape[2], hidden_size=128).to(device)

    hist = model.train_model(
        X_train, Y_train,
        target_len=target_len,
        batch_size=64,
        val_input_tensor=X_val,          
        val_target_tensor=Y_val,
        max_epochs=500,
        training_prediction="teacher_forcing",
        teacher_forcing_ratio=tfr,
        learning_rate=lr,
        patience=10,
        save_path=None
    )

    preds = rollout_recursive(model, X_test, target_len, batch_size=64)
    mae_m, mse_m2, rmse_m, per_step_mae, per_step_rmse = metrics_meters(preds, Y_test.to(device))

    model_path = out_dir / f"seq2seq_tlen{target_len}_tfr{tfr}_lr{lr}.pt"
    torch.save(model.state_dict(), model_path)

    rows.append({
        "target_len": target_len,
        "teacher_forcing_ratio": tfr,
        "learning_rate": lr,
        "MAE_m": mae_m,
        "MSE_m2": mse_m2,
        "RMSE_m": rmse_m,
        "best_val_loss": hist["best_val_loss"],
        "model_path": str(model_path),
        "per_step_MAE_m": ",".join(f"{x:.2f}" for x in per_step_mae),
        "per_step_RMSE_m": ",".join(f"{x:.2f}" for x in per_step_rmse),
    })

results_df = pd.DataFrame(rows)
results_df.sort_values("RMSE_m").head(10)


Combo: target_len=1, tfr=0.2, lr=0.01


  0%|          | 1/500 [00:10<1:23:41, 10.06s/it, no_improve=0, train=210.2323, val=0.3172]


KeyboardInterrupt: 

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda version:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.10.0+cpu
cuda version: None
cuda available: False
